# Regressão linear 

In [18]:
import pandas as pd
from pathlib import Path
import numpy as np
import statsmodels.formula.api as smf

Leitura do dataset.

In [19]:
caminho = Path(r'C:\Users\saulosgil\OneDrive\Documentos\consumo_alimentar_idososHosp\data\data_reg.xlsx')
df = pd.read_excel(caminho, engine='openpyxl', sheet_name= 'meta_ptn')

df.head()

,id,firstname,lastname,idade,sexo,NRS_2002,HAS,DMII,ICC,DRC,...,DoisGruposKcal,dias_internados,delta_csarf,delta_csavl,delta_hgs,delta_tug,delta_ts,delta_sppb,delta_barthel,cons_ptn_medio
0,35.0,Aderbal,Alves,99.0,Feminino,4.0,1.0,0.0,0.0,0.0,...,ProxMeta,21.0,-0.31,-0.74,0.0,1.99,15.55,-2.0,0.0,62.431619
1,6.0,Alvarina,Magro,75.0,Feminino,3.0,1.0,1.0,1.0,0.0,...,ProxMeta,12.0,-0.43,-1.00,2.0,-5.10,0.00,2.0,0.0,50.942667
2,37.0,Anisia,Rodrigues dos Santos,73.0,Feminino,4.0,1.0,1.0,0.0,0.0,...,ProxMeta,14.0,-0.19,-1.57,-2.0,0.63,1.94,-1.0,0.0,55.786071
3,29.0,Antonia,Bigoli,80.0,Feminino,3.0,1.0,1.0,0.0,0.0,...,ProxMeta,6.0,-1.16,-1.31,10.0,0.79,-1.53,-2.0,-5.0,72.751000
4,9.0,Arnaldo,Silva,77.0,Masculino,3.0,1.0,1.0,0.0,0.0,...,ProxMeta,6.0,-0.80,-0.18,-9.0,-0.18,-2.26,1.0,0.0,49.763333


Ajuste das variáveis para aplicar no modelo.

In [20]:
# Idade categorizada: '< 80 anos' ou '>= 80 anos' (idade ausente = NaN)
df['idade_cat'] = (pd.Series(np.where(df['idade'] < 80, '< 80 anos', '>= 80 anos'), index=df.index)
                   .where(df['idade'].notna()))

# Exposição em unidades de 10 gramas/dia: β = mudança no desfecho a cada +10 gramas/dia consumidas
df['ptn100'] = df['cons_ptn_medio'] / 100

# Regressão linear  

1. Desfechos
    - `AST VL`;
    - `AST RF`;                 
    - `HGS`;                  
    - `TUG`;        
    - `TST`; 
    - `SPPB`;        
    - `Barthel Index`;

In [21]:
ref = "ptn100"   # consumo proteico médio, por 100 ptn/dia

desfechos = {
    'delta_csarf':      'CSARF (cm²)',
    'delta_csavl':      'CSAVL (cm²)',
    'delta_hgs':        'HGS (kgf)',
    'delta_tug':        'TUG (seconds)',
    'delta_ts':         'TST (seconds)',
    'delta_sppb':       'SPPB (a.u.)',
    'delta_barthel':    'Barthel (a.u.)',
}

resultados_reg = []

for var, rotulo in desfechos.items():

    formula = f"{var} ~ {ref}"

    # Modelo de regressão linear com erros-padrão robustos HC3
    m = smf.ols(formula, data=df).fit(cov_type='HC3')

    b      = m.params[ref]                # β por +100 kcal/dia
    lo, hi = m.conf_int().loc[ref]        # IC 95%
    p      = m.pvalues[ref]

    resultados_reg.append({
        'Outcome': rotulo,
        'β por 10 gramas (IC 95%)': f'{b:.3f} ({lo:.3f} a {hi:.3f})',
        'P': round(p, 3),
        'n': int(m.nobs),
        'R² ajustado': round(m.rsquared_adj, 2),
    })

tabela_reg = pd.DataFrame(resultados_reg)
tabela_reg

,Outcome,β por 10 gramas (IC 95%),P,n,R² ajustado
0,CSARF (cm²),0.448 (-0.192 a 1.089),0.170,54,0.03
1,CSAVL (cm²),1.748 (0.097 a 3.399),0.038,54,0.08
2,HGS (kgf),5.404 (-1.549 a 12.356),0.128,54,0.03
3,TUG (seconds),2.037 (-0.878 a 4.953),0.171,54,-0.00
4,TST (seconds),-2.928 (-12.979 a 7.122),0.568,54,-0.01
5,SPPB (a.u.),0.568 (-2.422 a 3.557),0.710,54,-0.02
6,Barthel (a.u.),1.371 (-12.800 a 15.541),0.850,54,-0.02


# Regressão linear ajustada 

1. Desfechos
    - `AST VL`;
    - `AST RF`;                 
    - `HGS`;                  
    - `TUG`;        
    - `TST`; 
    - `SPPB`;        
    - `Barthel Index`;      

2. Covariáveis
    - `peso_adm`;
    - `sexo`;                 
    - `idade_cat`;                  
    - `NRS_2002`;        
    - `HAS`; 
    - `DMII`;        
    - `ICC`;   
    - `DRC`;        
    - `DPOC`;.      
    - `dias_internados`; 

In [22]:
ref = "ptn100"   # consumo proteico médio, por 100 ptn/dia

desfechos = {
    'delta_csarf':      'CSARF (cm²)',
    'delta_csavl':      'CSAVL (cm²)',
    'delta_hgs':        'HGS (kgf)',
    'delta_tug':        'TUG (seconds)',
    'delta_ts':         'TST (seconds)',
    'delta_sppb':       'SPPB (a.u.)',
    'delta_barthel':    'Barthel (a.u.)',
}

# Covariáveis: peso_adm e dias_internados entram como contínuas (sem C());
# as demais são categóricas/binárias.
covariaveis = ['peso_adm', 'dias_internados',
               'C(sexo)', 'C(idade_cat)', 'C(NRS_2002)',
               'C(DMII)', 'C(HAS)', 'C(ICC)', 'C(DRC)', 'C(DPOC)']

resultados_reg = []

for var, rotulo in desfechos.items():
    formula = f"{var} ~ {ref} + {' + '.join(covariaveis)}"
    m = smf.ols(formula, data=df).fit(cov_type='HC3')   # EP robustos, adequados para N pequeno

    b      = m.params[ref]                # β por +100 kcal/dia
    lo, hi = m.conf_int().loc[ref]        # IC 95%
    p      = m.pvalues[ref]

    resultados_reg.append({
        'Outcome': rotulo,
        'β por 10 gramas (IC 95%)': f'{b:.3f} ({lo:.3f} a {hi:.3f})',
        'P': round(p, 3),
        'n': int(m.nobs),
        'nº parâmetros': len(m.params),
        'R² ajustado': round(m.rsquared_adj, 2),
    })

tabela_reg = pd.DataFrame(resultados_reg)
tabela_reg

,Outcome,β por 10 gramas (IC 95%),P,n,nº parâmetros,R² ajustado
0,CSARF (cm²),0.511 (-0.354 a 1.375),0.247,54,13,-0.01
1,CSAVL (cm²),2.350 (0.501 a 4.199),0.013,54,13,0.19
2,HGS (kgf),5.349 (-5.202 a 15.900),0.320,54,13,0.00
3,TUG (seconds),1.167 (-3.519 a 5.853),0.626,54,13,-0.01
4,TST (seconds),-4.401 (-14.821 a 6.019),0.408,54,13,0.16
5,SPPB (a.u.),0.651 (-3.846 a 5.148),0.777,54,13,-0.05
6,Barthel (a.u.),1.299 (-22.798 a 25.397),0.916,54,13,0.05
